# Standortgründungen in der Steiermark

100-m-NB2-Hauptlauf für 2016–2025 aus RESULTS/100m/final/. Eingaben aus Notebook 01 und 08
unter ANAL/data/100m/. Der Export beginnt nach bestandenem Log-Likelihood-Abgleich.

3-km-Sechsecke nach Rasterzellmittelpunkt. Die Intensität bezieht sich auf die Fläche innerhalb
der Steiermark und ein Jahr. Beide Karten unterdrücken Sechsecke unter der Mindestzahl
beobachteter Gründungen oder Standorte. Die Quantilgrenzen werden für die Anzeige gerundet.

Der HTML-Export bettet die öffentliche Grundkarte ein und speichert ihren Cache in
work/founding_maps/. Für einen Rechner ohne Internet dort die WebP- und JSON-Datei bereitlegen.
[Grundkarte](https://basemap.at/) · [GIP](https://www.gip.gv.at/)

In [ ]:
from base64 import b64encode
from concurrent.futures import ThreadPoolExecutor
from html import escape
from io import BytesIO
import json
import math
from pathlib import Path
from string import Template
from tempfile import TemporaryDirectory
from xml.etree import ElementTree

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyogrio
from PIL import Image
import requests
from scipy.special import gammaln
import shapely

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ANAL").is_dir())
OUTPUT_DIR = PROJECT_DIR / "FIG" / "Map results" / "exports"
BASEMAP_AT = PROJECT_DIR / "work" / "founding_maps" / "basemap_at_steiermark.webp"

MINIMUM_COUNT = 5
GIP_PBF = None  # optionaler Pfad zu einem GIP2OSM-Export

START_YEAR, END_YEAR = 2016, 2025
CRS = "EPSG:3035"
QUARTERS = pd.period_range(f"{START_YEAR}Q1", f"{END_YEAR}Q4", freq="Q").astype(str)
RADIUS_M = 1500
RATIO_CUTS = np.array([0.67, 0.80, 0.95, 1.05, 1.25, 1.50])
SUPPRESSED = -1
DISTRICTS = {
    "601": "Graz (Stadt)", "603": "Deutschlandsberg", "606": "Graz-Umgebung",
    "610": "Leibnitz", "611": "Leoben", "612": "Liezen", "614": "Murau",
    "616": "Voitsberg", "617": "Weiz", "620": "Murtal", "621": "Bruck-Mürzzuschlag",
    "622": "Hartberg-Fürstenfeld", "623": "Südoststeiermark",
}

## 1. NB2-Hauptlauf prüfen

In [ ]:
def input_paths(project):
    data = project / "ANAL/data/100m"
    prepared = data / "modelldaten"
    results = project / "RESULTS/100m/final"
    paths = {
        "prepared": prepared / "model_data_info.json",
        "founding": prepared / "founding_frame.parquet",
        "raster": data / "raster_100m_styria.geoparquet",
        "locations": data / "firms_assigned_100m.geoparquet",
        "run": results / "laufinfo.json",
        "sample": results / "stichprobe.csv",
        "coefficients": results / "koeffizienten.csv",
        "diagnostics": results / "diagnostik.csv",
    }
    missing = [str(path.relative_to(project)) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError("Für den NB2-Hauptlauf fehlen lokal:\n" + "\n".join(missing))
    return paths


def load_model(paths):
    run = json.loads(paths["run"].read_text(encoding="utf-8"))
    prepared = json.loads(paths["prepared"].read_text(encoding="utf-8"))
    study = run.get("study_period", {})
    if ((run.get("resolution_m"), run.get("run_name")) != (100, "final")
            or (study.get("start_year"), study.get("end_year")) != (START_YEAR, END_YEAR)):
        raise ValueError("Erwartet wird ein abgeschlossener 100-m-Hauptlauf final für 2016–2025.")
    if (prepared.get("status"), prepared.get("resolution_m"), prepared.get("version")) != (
        "completed", 100, "model-tables-3"
    ):
        raise ValueError("Die Modellstichprobe muss aus Notebook 08 für 100 m stammen.")
    if (run.get("event_definition") and prepared.get("event_definition")
            and run["event_definition"] != prepared["event_definition"]):
        raise ValueError("Die Ereignisdefinitionen von Stichprobe und Modelllauf weichen ab.")

    coefficients = pd.read_csv(paths["coefficients"], float_precision="round_trip")
    main = coefficients.loc[coefficients.model.eq("founding_nb2")
                            & coefficients.family.eq("nb2") & coefficients.specification.eq("main")]
    beta = main.set_index("term").coefficient
    diagnostics = pd.read_csv(paths["diagnostics"], float_precision="round_trip")
    fit = diagnostics.loc[diagnostics.model.eq("founding_nb2") & diagnostics.check.eq("fit")
                          & diagnostics.status.eq("estimated") & diagnostics.family.eq("nb2")
                          & diagnostics.specification.eq("main")]
    sample = pd.read_csv(paths["sample"])
    sample = sample.loc[sample.model.eq("founding") & sample.stage.eq("model_sample")]
    if len(fit) != 1 or len(sample) != 1 or not beta.index.is_unique or not np.isfinite(beta).all():
        raise ValueError("Die Exporte müssen genau eine gültige NB2-Hauptschätzung enthalten.")
    fit, sample = fit.iloc[0], sample.iloc[0]

    terms = prepared["founding_terms"]
    period_terms = {f"period_{period}" for period in QUARTERS[1:]}
    mandatory = {"const", "alpha"} | period_terms
    if (not mandatory.issubset(beta.index) or set(beta.index) - mandatory - set(terms)
            or beta["alpha"] <= 0 or not np.isfinite(fit.log_likelihood)):
        raise ValueError("NB2-Konstante, Dispersion, Quartalseffekte oder Modellterme sind ungültig.")
    table = prepared["tables"]["founding"]
    if (table["file"] != paths["founding"].name or table["rows"] != sample.rows
            or fit.n_observations != sample.rows or fit.n_clusters != sample.units):
        raise ValueError("Stichprobenmetadaten passen nicht zum NB2-Hauptlauf.")
    return beta, terms, sample, float(fit.log_likelihood)


def score_founding(paths):
    """Reconstruct counts in memory and require exact sample and likelihood checks."""
    beta, terms, sample, reference_ll = load_model(paths)
    grid = pd.read_parquet(paths["raster"], columns=["grid_id", "centroid_x", "centroid_y"])
    grid_ids = pd.Index(grid.grid_id)
    if grid_ids.hasnans or not grid_ids.is_unique or not np.isfinite(grid[["centroid_x", "centroid_y"]]).all().all():
        raise ValueError("Das Raster muss eindeutige Zell-IDs und gültige EPSG:3035-Zentren enthalten.")
    observed = np.zeros(len(grid), dtype=np.int64)
    expected = np.zeros(len(grid))
    seen = np.zeros(len(grid) * len(QUARTERS), dtype=bool)

    data = pq.ParquetFile(paths["founding"])
    columns = ["grid_id", "period", "births", *terms]
    if data.metadata.num_rows != sample.rows or set(columns) - set(data.schema_arrow.names):
        raise ValueError("Zeilen oder Spalten der Gründungstabelle passen nicht zum Modelllauf.")
    period_effects = beta.reindex([f"period_{period}" for period in QUARTERS], fill_value=0).to_numpy()
    fitted_terms = [term for term in terms if term in beta.index]
    omitted_terms = [term for term in terms if term not in beta.index]
    omitted_values = None
    alpha = beta["alpha"]
    size, log_alpha = 1 / alpha, np.log(alpha)
    ll_parts = []
    for batch in data.iter_batches(batch_size=100_000, columns=columns):
        frame = batch.to_pandas()
        cells = grid_ids.get_indexer(frame.grid_id)
        periods = QUARTERS.get_indexer(frame.period)
        if (cells < 0).any() or (periods < 0).any():
            raise ValueError("Die Modellstichprobe enthält unbekannte Rasterzellen oder Quartale.")
        keys = cells * len(QUARTERS) + periods
        if seen[keys].any() or len(np.unique(keys)) != len(keys):
            raise ValueError("Die Modellstichprobe enthält doppelte Zell-Quartals-Schlüssel.")
        seen[keys] = True
        y = frame.births.to_numpy(dtype=float)
        predictors = frame[fitted_terms].to_numpy(dtype=float)
        if (not np.isfinite(y).all() or (y < 0).any() or (y != np.floor(y)).any()
                or not np.isfinite(predictors).all()):
            raise ValueError("Die Modellstichprobe enthält ungültige Zählwerte oder Regressoren.")
        if omitted_terms:
            omitted = frame[omitted_terms].to_numpy(dtype=float)
            if omitted_values is None:
                omitted_values = omitted[0]
            if not np.isfinite(omitted_values).all() or (omitted != omitted_values).any():
                raise ValueError("Ein im NB2-Export fehlender Modellterm ist nicht konstant.")

        eta = beta["const"] + period_effects[periods] + predictors @ beta[fitted_terms].to_numpy()
        with np.errstate(over="ignore", under="ignore", invalid="ignore"):
            mu = np.exp(eta)
            log_odds = log_alpha + eta
            loglik = (gammaln(y + size) - gammaln(size) - gammaln(y + 1)
                      + y * log_odds - (y + size) * np.logaddexp(0, log_odds))
        if not np.isfinite(loglik).all() or not (np.isfinite(mu) & (mu > 0)).all():
            raise ValueError("Die rekonstruierten NB2-Vorhersagen oder Likelihood-Beiträge sind ungültig.")
        ll_parts.append(float(loglik.sum()))
        np.add.at(observed, cells, y.astype(np.int64))
        np.add.at(expected, cells, mu)

    support = seen.reshape(len(grid), len(QUARTERS))
    retained = support.any(axis=1)
    if (not support.any(axis=0).all() or retained.sum() != sample.units
            or observed.sum() != sample.events):
        raise ValueError("Quartale, Rasterzellen oder beobachtete Gründungen weichen vom Hauptlauf ab.")
    difference = abs(math.fsum(ll_parts) - reference_ll)
    tolerance = 1e-5 + 1e-9 * abs(reference_ll)
    if difference > tolerance:
        raise ValueError(f"NB2-Log-Likelihood weicht ab (|Δ|={difference:.6g}, Toleranz={tolerance:.6g}). "
                         "Verwende die Modellstichprobe des gespeicherten Hauptlaufs.")

    scored = grid.loc[retained].copy()
    scored["observed"] = observed[retained]
    scored["expected"] = expected[retained]
    sites = pd.read_parquet(paths["locations"], columns=["firm_id", "grid_id", "founding_date", "exit_date"])
    if sites.firm_id.isna().any() or sites.firm_id.duplicated().any() or not sites.grid_id.isin(grid_ids).all():
        raise ValueError("Standort-IDs oder ihre Rasterzuordnung sind ungültig.")
    present = (sites.founding_date.lt(pd.Timestamp(END_YEAR + 1, 1, 1))
               & (sites.exit_date.isna() | sites.exit_date.ge(pd.Timestamp(START_YEAR, 1, 1))))
    counted = sites.loc[present].groupby("grid_id").size()
    scored["locations"] = scored.grid_id.map(counted).fillna(0).astype("int64")
    if scored.observed.gt(scored.locations).any():
        raise ValueError("Beobachtete Gründungen passen nicht zu den Standortdaten.")
    return scored, difference, tolerance

## 2. Sechsecke klassieren

In [ ]:
def load_basemap(project, gip_path=None):
    archive = (project / "OGD/Gemeindegrenzen.zip").resolve()
    if not archive.is_file():
        raise FileNotFoundError("Die Gemeindegrenzen fehlen unter OGD/Gemeindegrenzen.zip.")
    municipalities = gpd.read_file(f"zip://{archive.as_posix()}", columns=["BEZNR6"]).to_crs(CRS)
    municipalities["district"] = municipalities.BEZNR6.astype("string").map(DISTRICTS)
    if municipalities.district.isna().any():
        raise ValueError("Ungültige Bezirkskennziffern in den Gemeindegrenzen.")
    districts = municipalities[["district", "geometry"]].dissolve(by="district", as_index=False)
    state = districts.geometry.union_all()

    default_gip = project / "OGD/GIP/2025.osm.pbf"
    if gip_path is None:
        adjacent_gip = project.parent / "GIP2OSM/austria-gip2.osm.pbf"
        path = next((candidate for candidate in (default_gip, adjacent_gip) if candidate.is_file()), default_gip)
    else:
        path = (project / gip_path).resolve()
    if not path.is_file():
        raise FileNotFoundError(f"GIP-Straßennetz fehlt unter {path}. Setze GIP_PBF auf einen GIP2OSM-Export.")
    bounds_wgs84 = districts.to_crs(4326).total_bounds
    roads = pyogrio.read_dataframe(path, layer="lines", columns=["highway"], bbox=tuple(bounds_wgs84),
                                   where="highway = 'motorway'")
    roads = roads.to_crs(CRS)
    roads = gpd.clip(roads, state, keep_geom_type=True)
    if roads.empty:
        raise ValueError("Der GIP-Export enthält keine Autobahnlinien in der Steiermark.")
    district_boundaries = districts.boundary.union_all()
    municipality_boundaries = municipalities.boundary.union_all()
    return {
        "state": state, "districts": districts,
        "district_lines": shapely.line_merge(district_boundaries.difference(state.boundary)).simplify(30),
        "municipality_lines": shapely.line_merge(municipality_boundaries.difference(district_boundaries)).simplify(38),
        "motorways": shapely.line_merge(roads.geometry.union_all()).simplify(25),
        "gip_path": path,
    }


def hex_indices(x, y):
    """Round centroids to the fixed flat-topped hex lattice in EPSG:3035."""
    q = (2 / 3) * np.asarray(x) / RADIUS_M
    r = (np.asarray(y) / np.sqrt(3) - np.asarray(x) / 3) / RADIUS_M
    cube = np.column_stack([q, r, -q - r])
    rounded = np.rint(cube)
    largest = np.argmax(np.abs(cube - rounded), axis=1)
    rounded[np.arange(len(rounded)), largest] -= rounded.sum(axis=1)
    return rounded[:, :2].astype(np.int64)


def hex_grid(state):
    xmin, ymin, xmax, ymax = state.bounds
    corners = hex_indices([xmin, xmin, xmax, xmax], [ymin, ymax, ymin, ymax])
    columns, rows = np.meshgrid(
        np.arange(corners[:, 0].min() - 1, corners[:, 0].max() + 2),
        np.arange(corners[:, 1].min() - 1, corners[:, 1].max() + 2),
    )
    q, r = columns.ravel(), rows.ravel()
    centers = RADIUS_M * np.column_stack([1.5 * q, np.sqrt(3) * (r + q / 2)])
    angles = np.arange(6) * np.pi / 3
    offsets = RADIUS_M * np.column_stack([np.cos(angles), np.sin(angles)])
    polygons = shapely.polygons(shapely.linearrings(centers[:, None, :] + offsets))
    shapely.prepare(state)
    keep = shapely.intersects(state, polygons)
    polygons, q, r = polygons[keep], q[keep], r[keep]
    crossing = ~shapely.covers(state, polygons)
    polygons[crossing] = shapely.intersection(polygons[crossing], state)
    return gpd.GeoDataFrame({"q": q, "r": r}, geometry=polygons, crs=CRS).loc[
        shapely.area(polygons) > 0
    ].reset_index(drop=True)


def assign_hexes(cells, hexes):
    pairs = pd.MultiIndex.from_frame(hexes[["q", "r"]])
    qs, rs = hex_indices(cells.centroid_x, cells.centroid_y).T
    assigned = pairs.get_indexer(pd.MultiIndex.from_arrays([qs, rs]))
    outside = np.flatnonzero(assigned < 0)
    if len(outside):
        # A border raster cell can have its centroid outside the clipped lattice.
        x, y = cells.centroid_x.to_numpy()[outside], cells.centroid_y.to_numpy()[outside]
        footprints = gpd.GeoDataFrame(
            geometry=shapely.box(x - 50, y - 50, x + 50, y + 50), index=outside, crs=CRS,
        )
        candidates = footprints.sjoin(hexes[["geometry"]])
        overlap = shapely.area(shapely.intersection(
            candidates.geometry.array, hexes.geometry.iloc[candidates.index_right].array
        ))
        selected = candidates.assign(overlap=overlap).loc[lambda frame: frame.overlap.gt(0)]
        selected = selected.sort_values(["overlap", "index_right"], ascending=[False, True])
        selected = selected.loc[~selected.index.duplicated()]
        assigned[selected.index.to_numpy()] = selected.index_right.to_numpy()
    if (assigned < 0).any():
        raise ValueError("Mindestens eine Modellzelle lässt sich keinem Sechseck in der Steiermark zuordnen.")
    return assigned


def intensity_class_labels(cuts):
    """Round the public cut points, after assigning classes with full precision."""
    digits = 2
    distinct = len(np.unique(cuts))
    while digits < 4 and len(np.unique(np.round(cuts, digits))) < distinct:
        digits += 1
    formatted = [f"{cut:.{digits}f}".replace(".", ",") for cut in cuts]
    middle = [f"{lower}–{upper}" if lower != upper else f"≈ {lower}"
              for lower, upper in zip(formatted, formatted[1:])]
    return [f"≤ {formatted[0]}", *middle, f"> {formatted[-1]}"]


def classify_hexes(cells, basemap, minimum_count=5):
    """Return classified model hexes and rounded legend labels, including a suppression class."""
    if not isinstance(minimum_count, int) or isinstance(minimum_count, bool) or minimum_count < 1:
        raise ValueError("Die Mindestzahl muss eine positive ganze Zahl sein.")
    hexes = hex_grid(basemap["state"])
    assigned = assign_hexes(cells, hexes)
    totals = cells[["observed", "expected", "locations"]].groupby(assigned).sum()
    allowed = totals.loc[(totals.observed >= minimum_count) & (totals.locations >= minimum_count)]
    mapped = hexes.loc[totals.index, ["geometry"]].copy()
    mapped["intensity"] = SUPPRESSED
    mapped["ratio"] = SUPPRESSED
    labels = []
    if not allowed.empty:
        # Use the in-state part of each hexagon, with no extrapolation to missing cell-quarters.
        area_km2 = mapped.loc[allowed.index].area / 1_000_000
        intensity = allowed.expected / area_km2 / (END_YEAR - START_YEAR + 1)
        cuts = np.quantile(intensity, np.arange(1, 6) / 6)
        mapped.loc[allowed.index, "intensity"] = np.searchsorted(cuts, intensity, side="left")
        mapped.loc[allowed.index, "ratio"] = np.searchsorted(
            RATIO_CUTS, allowed.observed / allowed.expected, side="right"
        )
        labels = intensity_class_labels(cuts)

    districts = basemap["districts"].reset_index(drop=True)
    overlaps = mapped[["geometry"]].sjoin(districts)
    area = shapely.area(shapely.intersection(overlaps.geometry.array, districts.geometry.iloc[overlaps.index_right].array))
    dominant = overlaps.assign(overlap=area).loc[lambda frame: frame.overlap.gt(0)]
    dominant = dominant.sort_values(["overlap", "district"], ascending=[False, True])
    mapped["district"] = dominant.district.loc[~dominant.index.duplicated()]
    if mapped.district.isna().any():
        raise ValueError("Mindestens ein Sechseck mit Modellzellen hat keine Bezirkszuordnung.")
    return mapped[["district", "intensity", "ratio", "geometry"]], labels

## 3. Druckkarten

In [ ]:
INTENSITY = ("#d6e4eb", "#b4cfda", "#87aebf", "#5f90a4", "#3d687c", "#204658")
RATIO = ("#9d482c", "#c9794b", "#ebbc91", "#ebe8e0", "#aeced6", "#5997a8", "#266375")
BACKGROUND = "#f5f5f3"
SUPPRESSION_FILL = "#a4a4a4"
STATE_LINE = "#303a3d"
DISTRICT_LINE = "#70797c"
MUNICIPALITY_LINE = "#a2aaac"
ROAD_LINE = "#6b3e6d"
SUPPRESSION_LABEL = "unterdrückt (zu wenige Fälle)"
HTML_CRS = "EPSG:31256"

CUT_LABELS = [f"{cut:.2f}".replace(".", ",") for cut in RATIO_CUTS]
RATIO_LABELS = (
    [f"< {CUT_LABELS[0]}"]
    + [f"{left} bis < {right}" for left, right in zip(CUT_LABELS, CUT_LABELS[1:])]
    + [f"≥ {CUT_LABELS[-1]}"]
)
INTENSITY_LEGEND = "Erwartete Gründungen je km² und Jahr"
RATIO_LEGEND = "Beobachtete / erwartete Gründungen"
PREVIEW = "Synthetische Vorschau"


def map_bounds(state):
    xmin, ymin, xmax, ymax = state.bounds
    return xmin - 2800, ymin - 2800, xmax + 2800, ymax + 2800


def draw_map(ax, hexes, basemap, layer):
    state = gpd.GeoSeries([basemap["state"].simplify(25)], crs=CRS)
    state.plot(ax=ax, color=BACKGROUND, zorder=1)
    colors = INTENSITY if layer == "intensity" else RATIO
    suppressed = hexes[layer].eq(SUPPRESSED)
    if suppressed.any():
        hexes.loc[suppressed].plot(ax=ax, color=SUPPRESSION_FILL, edgecolor="#6d6d6d",
                                   linewidth=.16, hatch="////", zorder=2)
    released = hexes.loc[~suppressed]
    if len(released):
        released.plot(ax=ax, color=np.take(colors, released[layer]), edgecolor="white", linewidth=.16, zorder=2)
    gpd.GeoSeries([basemap["municipality_lines"]], crs=CRS).plot(
        ax=ax, color=MUNICIPALITY_LINE, linewidth=.19, alpha=.58, zorder=3
    )
    gpd.GeoSeries([basemap["district_lines"]], crs=CRS).plot(ax=ax, color=DISTRICT_LINE, linewidth=.34, zorder=4)
    roads = gpd.GeoSeries([basemap["motorways"]], crs=CRS)
    roads.plot(ax=ax, color="white", linewidth=1.15, zorder=5)
    roads.plot(ax=ax, color=ROAD_LINE, linewidth=.65, zorder=6)
    state.boundary.plot(ax=ax, color=STATE_LINE, linewidth=.67, zorder=7)

    xmin, ymin, xmax, ymax = map_bounds(basemap["state"])
    scale_x, scale_y = xmin + .035 * (xmax - xmin), ymin + .045 * (ymax - ymin)
    scale_length = 20_000
    ax.plot([scale_x, scale_x + scale_length], [scale_y, scale_y], color=STATE_LINE, linewidth=.9)
    ax.plot([scale_x, scale_x + scale_length], [scale_y, scale_y], "|", color=STATE_LINE, markersize=3.4)
    ax.annotate("20 km", (scale_x + scale_length / 2, scale_y), xytext=(0, 3), textcoords="offset points",
                ha="center", fontsize=6.4, color=STATE_LINE)
    ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax), aspect="equal")
    ax.set_axis_off()


def intensity_key(ax, labels, columns):
    ax.set_axis_off()
    ax.text(0, 1, INTENSITY_LEGEND, va="top", fontsize=8.2, color=STATE_LINE)
    if not labels:
        ax.text(0, .39, "Keine freigegebenen Sechsecke", fontsize=7.4)
        return
    for index, (color, label) in enumerate(zip(INTENSITY, labels)):
        x = (index % columns) / columns
        y = .40 if columns == 6 else .50 - .27 * (index // columns)
        width = .025 if columns == 6 else .046
        ax.add_patch(Rectangle((x, y - .08), width, .16, facecolor=color,
                               edgecolor="#a9b2b3", linewidth=.35))
        ax.text(x + width + .012, y, label, va="center", fontsize=7.15, color=STATE_LINE)


def ratio_key(ax):
    ax.set_axis_off()
    ax.text(0, 1, RATIO_LEGEND, va="top", fontsize=8.2, color=STATE_LINE)
    width = 1 / len(RATIO)
    for index, color in enumerate(RATIO):
        ax.add_patch(Rectangle((index * width, .40), width, .16, facecolor=color,
                               edgecolor="#a9b2b3", linewidth=.35))
    for index, cut in enumerate(CUT_LABELS, start=1):
        ax.text(index * width, .30, cut, ha="center", va="top", fontsize=7.15)


def figure_notes(fig, y, preview):
    keys = [
        Line2D([], [], color=ROAD_LINE, linewidth=.8, label="GIP-Autobahn"),
        Patch(facecolor=SUPPRESSION_FILL, edgecolor="#6d6d6d", linewidth=.4,
              hatch="////", label=SUPPRESSION_LABEL),
        Patch(facecolor=BACKGROUND, edgecolor="#c8cccd", linewidth=.4, label="Ohne Modellzellen"),
    ]
    fig.legend(handles=keys, ncol=3, loc="center", bbox_to_anchor=(.5, y),
               fontsize=7, columnspacing=1.55, handlelength=1.35, frameon=False)
    fig.text(.028, .024, "Grenzen: Land Steiermark · Datenquelle: gip.gv.at",
             fontsize=6.5, color="#536065", url="https://www.gip.gv.at/")
    if preview:
        fig.text(.972, .024, PREVIEW, ha="right", fontsize=6.5, color="#536065")


def print_figures(hexes, basemap, intensity_labels, *, preview=False):
    with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": ["Arial", "DejaVu Sans"],
                         "hatch.linewidth": .32, "figure.facecolor": "white"}):
        figure_a = plt.figure(figsize=(16 / 2.54, 11 / 2.54))
        draw_map(figure_a.add_axes([.050, .157, .90, .825]), hexes, basemap, "intensity")
        intensity_key(figure_a.add_axes([.094, .080, .812, .074]), intensity_labels, columns=6)
        figure_notes(figure_a, y=.062, preview=preview)

        figure_b = plt.figure(figsize=(16 / 2.54, 8 / 2.54))
        draw_map(figure_b.add_axes([.023, .238, .462, .741]), hexes, basemap, "intensity")
        draw_map(figure_b.add_axes([.515, .238, .462, .741]), hexes, basemap, "ratio")
        intensity_key(figure_b.add_axes([.038, .103, .432, .135]), intensity_labels, columns=3)
        ratio_key(figure_b.add_axes([.530, .103, .432, .135]))
        figure_notes(figure_b, y=.076, preview=preview)

    return {"figure_A_gruendungsintensitaet": figure_a, "figure_B_intensitaet_und_verhaeltnis": figure_b}

## 4. Interaktive Karte

In [ ]:
WMTS_URL = "https://mapsneu.wien.gv.at/basemap31256neu/1.0.0/WMTSCapabilities.xml"


def cache_basemap(path, bounds):
    """Crop and cache unlabelled basemap.at tiles using only the public map extent."""
    metadata_path = path.with_suffix(".json")
    if path.is_file() and metadata_path.is_file():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        left, bottom, right, top = metadata["bounds"]
        xmin, ymin, xmax, ymax = bounds
        if metadata["crs"] == HTML_CRS and left <= xmin < xmax <= right and bottom <= ymin < ymax <= top:
            return metadata

    try:
        response = requests.get(WMTS_URL, timeout=25)
        response.raise_for_status()
        contents = ElementTree.fromstring(response.content).find("{*}Contents")
        layer = next(item for item in contents.findall("{*}Layer")
                     if item.findtext("{*}Identifier") == "bmapstumm_grau")
        layer_url = layer.find("{*}ResourceURL").attrib["template"]
        matrix_set = next(item for item in contents.findall("{*}TileMatrixSet")
                          if item.findtext("{*}Identifier") == "31256")
        matrix = next(item for item in matrix_set.findall("{*}TileMatrix")
                      if item.findtext("{*}Identifier") == "10")
        # EPSG:31256 lists northing before easting in the WMTS capabilities.
        origin_y, origin_x = map(float, matrix.findtext("{*}TopLeftCorner").split())
        metres = float(matrix.findtext("{*}ScaleDenominator")) * .00028
        width, height = (int(matrix.findtext(f"{{*}}Tile{side}")) for side in ("Width", "Height"))
        xmin, ymin, xmax, ymax = bounds
        first_col = math.floor((xmin - origin_x) / (width * metres))
        last_col = math.floor((xmax - origin_x) / (width * metres))
        first_row = math.floor((origin_y - ymax) / (height * metres))
        last_row = math.floor((origin_y - ymin) / (height * metres))
        addresses = [(row, col) for row in range(first_row, last_row + 1)
                     for col in range(first_col, last_col + 1)]

        def fetch_tile(address):
            row, col = address
            url = layer_url.format(Style="grau", TileMatrixSet="31256", TileMatrix="10",
                                   TileRow=row, TileCol=col)
            response = requests.get(url, timeout=25)
            response.raise_for_status()
            with Image.open(BytesIO(response.content)) as tile:
                return tile.convert("RGBA")

        mosaic = Image.new("RGB", ((last_col - first_col + 1) * width, (last_row - first_row + 1) * height), "white")
        with ThreadPoolExecutor(max_workers=4) as pool:
            for (row, col), tile in zip(addresses, pool.map(fetch_tile, addresses)):
                mosaic.paste(tile, ((col - first_col) * width, (row - first_row) * height), tile)
    except requests.RequestException as error:
        raise ConnectionError(f"Grundkarte nicht verfügbar. Cache unter {metadata_path.parent} bereitlegen.") from error

    left = origin_x + first_col * width * metres
    top = origin_y - first_row * height * metres
    crop = (math.floor((xmin - left) / metres), math.floor((top - ymax) / metres),
            math.ceil((xmax - left) / metres), math.ceil((top - ymin) / metres))
    metadata = {
        "crs": HTML_CRS,
        "bounds": [left + crop[0] * metres, top - crop[3] * metres,
                   left + crop[2] * metres, top - crop[1] * metres],
        "source": "basemap.at STUMM GRAU, WMTS bmapstumm_grau, TileMatrixSet 31256, TileMatrix 10",
        "source_url": "https://basemap.at/",
        "license": "CC BY 4.0",
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(prefix=".basemap-", dir=path.parent) as staging:
        image_file = Path(staging) / path.name
        info_file = Path(staging) / metadata_path.name
        mosaic.crop(crop).save(image_file, quality=84, method=6)
        info_file.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
        image_file.replace(path)
        info_file.replace(metadata_path)
    return metadata

In [ ]:
HTML_TEMPLATE = Template(r"""
<!doctype html>
<html lang="de">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Standortgründungen in der Steiermark</title>
<style>
  :root { color-scheme: light; color: #25353c; background: white; font: 16px/1.45 Arial, sans-serif; }
  body { margin: 0; }
  main { max-width: 1260px; margin: auto; padding: clamp(14px, 3vw, 36px); }
  h1 { font-size: clamp(23px, 3vw, 34px); line-height: 1.1; margin: 0 0 8px; letter-spacing: -.03em; }
  header p { color: #4b6169; margin: 0; }
  fieldset { border: 0; padding: 0; margin: 16px 0 10px; display: flex; flex-wrap: wrap; gap: 8px; }
  fieldset label { position: relative; cursor: pointer; }
  fieldset input, .visually-hidden {
    position: absolute; width: 1px; height: 1px; overflow: hidden; clip-path: inset(50%);
  }
  fieldset span { display: block; border: 1px solid #a5b5ba; border-radius: 6px; padding: 8px 14px; background: white; }
  fieldset input:checked + span { color: white; background: #204658; border-color: #204658; }
  fieldset input:focus-visible + span, button:focus-visible, svg:focus-visible { outline: 2px solid #32728b; outline-offset: 3px; }
  .card { display: grid; grid-template-columns: minmax(0, 1fr) 235px; background: white; border: 1px solid #d9e2e4; }
  .map-frame { position: relative; min-width: 0; display: grid; align-items: center; padding: 12px; overflow: hidden; }
  svg { width: 100%; height: auto; display: block; touch-action: none; cursor: grab; overflow: hidden; }
  svg:active { cursor: grabbing; }
  svg path { vector-effect: non-scaling-stroke; }
  .hex { stroke: white; stroke-width: .45; cursor: crosshair; }
  .hex:hover { stroke: #143b49; stroke-width: 1.25; }
  #grundkarte { opacity: .70; }
  #map .hex[data-intensity="-1"] { fill: $suppression_fill; }
  $color_rules
  .zoom { position: absolute; top: 18px; left: 18px; display: flex; gap: 4px; }
  button {
    font: bold 16px Arial, sans-serif; line-height: 1; color: #204658; background: white; border: 1px solid #a5b5ba;
    border-radius: 5px; width: 34px; height: 34px; cursor: pointer;
  }
  button:hover { background: #eef4f5; }
  #tooltip { position: absolute; z-index: 2; background: #203940; color: white; border-radius: 5px; font-size: 13px;
    max-width: calc(100% - 34px); padding: 7px 10px; white-space: pre-line; pointer-events: none; }
  aside { padding: 16px; border-left: 1px solid #d9e2e4; font-size: 14px; }
  aside h2 { font-size: 15px; line-height: 1.34; margin: 0 0 10px; }
  aside p { color: #4b6169; margin: 12px 0 0; }
  ul { display: grid; gap: 6px; padding: 0; margin: 0; list-style: none; }
  li { display: flex; gap: 9px; align-items: center; }
  .swatch { width: 18px; height: 13px; box-sizing: border-box; border: 1px solid #aebbbd; background: var(--swatch); }
  .line { width: 18px; border-top: 2px solid var(--swatch); }
  .context { border-top: 1px solid #d9e2e4; margin-top: 15px; padding-top: 14px; }
  footer { margin-top: 13px; font-size: 13px; color: #4b6169; }
  footer a { color: #295769; }
  [hidden] { display: none !important; }
  @media (max-width: 720px) {
    .card { grid-template-columns: minmax(0, 1fr); }
    aside { border-left: 0; border-top: 1px solid #d9e2e4; padding: 16px; }
    ul { grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); }
    .zoom { top: 8px; left: 8px; }
  }
</style>
</head>
<body>
<main>
  <header>
    <h1>Standortgründungen in der Steiermark</h1>
    <p>$subtitle</p>
  </header>
  <fieldset>
    <legend class="visually-hidden">Kartenebene</legend>
    <label><input name="layer" type="radio" value="intensity" checked><span>Erwartet</span></label>
    <label><input name="layer" type="radio" value="ratio"><span>Beobachtet / erwartet</span></label>
  </fieldset>
  <div class="card">
    <div class="map-frame">
      <svg id="map" data-layer="intensity" viewBox="$viewbox" role="img"
           aria-label="Karte der Steiermark. Pfeiltasten verschieben die Karte, Plus und Minus ändern den Zoom." tabindex="0">
        $basemap_at
        <path d="$state" fill="$background" fill-opacity=".46" fill-rule="evenodd"/>
        <g fill-rule="evenodd">
          $polygons
        </g>
        <g fill="none" pointer-events="none" stroke-linecap="round" stroke-linejoin="round">
          <path id="gemeindegrenzen" d="$municipalities" stroke="$municipality_line" stroke-width=".62"/>
          <path d="$borders" stroke="$district_line" stroke-width="1.05"/>
          <path d="$roads" stroke="white" stroke-width="3.25"/>
          <path id="gip-autobahnen" d="$roads" stroke="$road_line" stroke-width="1.75"/>
          <path d="$state" stroke="$state_line" stroke-width="1.4"/>
        </g>
      </svg>
      <div class="zoom" aria-label="Kartenzoom">
        <button type="button" data-zoom=".75" aria-label="Vergrößern">+</button>
        <button type="button" data-zoom="1.333333" aria-label="Verkleinern">−</button>
        <button type="button" id="reset" aria-label="Gesamtansicht">↺</button>
      </div>
      <div role="tooltip" id="tooltip" hidden></div>
    </div>
    <aside aria-label="Kartenlegende">
      <section id="intensity-legend">
        <h2>$intensity_title</h2>
        <ul>$intensity_legend</ul>
        $empty_intensity
      </section>
      <section id="ratio-legend" hidden>
        <h2>$ratio_title</h2>
        <ul>$ratio_legend</ul>
      </section>
      <ul class="context">
        <li><span class="line" style="--swatch:$road_line"></span>GIP-Autobahn</li>
        <li id="suppressed-label"><span class="swatch" style="--swatch:$suppression_fill"></span>$suppression_label</li>
        <li><span class="swatch" style="--swatch:$background"></span>Ohne Modellzellen</li>
      </ul>
    </aside>
  </div>
  <footer>Grenzen: Land Steiermark · Datenquelle: <a href="https://www.gip.gv.at/">gip.gv.at</a> · Grundkarte: <a href="https://basemap.at/">basemap.at</a></footer>
</main>
<script>
  const map = document.querySelector('#map');
  const tooltip = document.querySelector('#tooltip');
  const suppressionLabel = document.querySelector('#suppressed-label').textContent.trim();
  const frame = map.parentElement;
  window.addEventListener('resize', () => { tooltip.hidden = true; });
  const view = map.viewBox.baseVal;
  const home = { width: view.width, height: view.height };
  const legends = {
    intensity: document.querySelector('#intensity-legend'),
    ratio: document.querySelector('#ratio-legend'),
  };
  const labels = Object.fromEntries(Object.entries(legends).map(([layer, legend]) => [
    layer, Array.from(legend.querySelectorAll('li'), item => item.textContent.trim()),
  ]));

  for (const input of document.querySelectorAll('input[name=layer]')) {
    input.addEventListener('change', () => {
      map.dataset.layer = input.value;
      for (const [layer, legend] of Object.entries(legends)) legend.hidden = layer !== input.value;
      tooltip.hidden = true;
    });
  }

  function setView(x, y, width) {
    const height = home.height * width / home.width;
    view.x = Math.max(0, Math.min(home.width - width, x));
    view.y = Math.max(0, Math.min(home.height - height, y));
    view.width = width;
    view.height = height;
    tooltip.hidden = true;
  }

  function zoom(factor, x = view.x + view.width / 2, y = view.y + view.height / 2) {
    const width = Math.min(home.width, Math.max(home.width / 8, view.width * factor));
    const scale = width / view.width;
    setView(x - (x - view.x) * scale, y - (y - view.y) * scale, width);
  }

  for (const button of document.querySelectorAll('[data-zoom]')) {
    button.addEventListener('click', () => zoom(Number(button.dataset.zoom)));
  }
  document.querySelector('#reset').addEventListener('click', () => setView(0, 0, home.width));

  function mapPoint(event) {
    return new DOMPoint(event.clientX, event.clientY).matrixTransform(map.getScreenCTM().inverse());
  }

  map.addEventListener('wheel', event => {
    event.preventDefault();
    const point = mapPoint(event);
    zoom(event.deltaY < 0 ? .81 : 1.24, point.x, point.y);
  }, { passive: false });

  let drag;
  function showTooltip(event, hex) {
    if (!hex) { tooltip.hidden = true; return; }
    const layer = map.dataset.layer;
    const index = Number(hex.dataset[layer]);
    tooltip.textContent = hex.dataset.bezirk + '\n' + (index < 0 ? suppressionLabel : labels[layer][index]);
    tooltip.hidden = false;
    const rect = frame.getBoundingClientRect();
    const x = event.clientX - rect.left + 13;
    const y = event.clientY - rect.top + 13;
    tooltip.style.left = Math.max(5, Math.min(x, rect.width - tooltip.offsetWidth - 5)) + 'px';
    tooltip.style.top = Math.max(5, Math.min(y, rect.height - tooltip.offsetHeight - 5)) + 'px';
  }

  map.addEventListener('pointerdown', event => {
    if (event.button !== 0) return;
    drag = { point: mapPoint(event), hex: event.target.closest('.hex'), moved: false };
    map.setPointerCapture(event.pointerId);
  });
  map.addEventListener('pointermove', event => {
    if (drag && view.width < home.width) {
      const point = mapPoint(event);
      drag.moved = true;
      setView(view.x + drag.point.x - point.x, view.y + drag.point.y - point.y, view.width);
    } else if (event.pointerType !== 'touch') {
      showTooltip(event, event.target.closest('.hex'));
    }
  });
  map.addEventListener('pointerup', event => {
    if (drag && !drag.moved) showTooltip(event, drag.hex);
    drag = undefined;
  });
  map.addEventListener('pointercancel', () => { drag = undefined; tooltip.hidden = true; });
  map.addEventListener('pointerleave', () => { tooltip.hidden = true; });
  map.addEventListener('keydown', event => {
    const moves = { ArrowLeft: [-1, 0], ArrowRight: [1, 0], ArrowUp: [0, -1], ArrowDown: [0, 1] };
    if (Object.hasOwn(moves, event.key)) {
      const [x, y] = moves[event.key];
      setView(view.x + x * view.width / 8, view.y + y * view.height / 8, view.width);
    } else if (event.key === '+' || event.key === '-') {
      zoom(event.key === '+' ? .75 : 1.333333);
    } else {
      return;
    }
    event.preventDefault();
  });
</script>
</body>
</html>
""")

In [ ]:
def svg_path(geometry, xy):
    if geometry.is_empty or geometry.geom_type == "Point":
        return ""
    if geometry.geom_type == "Polygon":
        return " ".join(svg_path(ring, xy) for ring in [geometry.exterior, *geometry.interiors])
    if geometry.geom_type in {"LineString", "LinearRing"}:
        coordinates = " ".join(f"{x:.3f},{y:.3f}" for x, y in (xy(x, y) for x, y in geometry.coords))
        return "M " + coordinates + (" Z" if geometry.geom_type == "LinearRing" else "")
    return " ".join(svg_path(part, xy) for part in geometry.geoms)


def html_legend(colors, labels):
    return "\n".join(
        f'<li><span class="swatch" style="--swatch:{color}"></span>{escape(label)}</li>'
        for color, label in zip(colors, labels)
    )


def embedded_basemap(path, view_bounds, xy):
    metadata = cache_basemap(path, view_bounds)
    left, bottom, right, top = metadata["bounds"]
    xmin, ymin, xmax, ymax = view_bounds
    if metadata["crs"] != HTML_CRS or not (left <= xmin < xmax <= right and bottom <= ymin < ymax <= top):
        raise ValueError("Der eingebettete basemap.at-Ausschnitt deckt das Kartengebiet nicht ab.")
    x, y = xy(left, top)
    content = b64encode(path.read_bytes()).decode("ascii")
    return (f'<image id="grundkarte" x="{x:.3f}" y="{y:.3f}" width="{(right-left)/1000:.3f}" '
            f'height="{(top-bottom)/1000:.3f}" preserveAspectRatio="none" '
            f'href="data:image/webp;base64,{content}" pointer-events="none"/>')


def write_interactive(hexes, basemap, intensity_labels, path, *, preview=False, background_path=BASEMAP_AT):
    keys = ["state", "municipality_lines", "district_lines", "motorways"]
    context = gpd.GeoSeries([basemap[key] for key in keys], index=keys, crs=CRS).to_crs(HTML_CRS)
    hexes = hexes.to_crs(HTML_CRS)
    bounds = map_bounds(context["state"])
    xmin, ymin, xmax, ymax = bounds

    def xy(x, y):
        return (x - xmin) / 1000, (ymax - y) / 1000

    state = svg_path(context["state"].simplify(25), xy)
    borders = svg_path(context["district_lines"], xy)
    municipalities = svg_path(context["municipality_lines"], xy)
    roads = svg_path(context["motorways"], xy)
    # Hexes carry the class codes and district, while the legend contains rounded cut points.
    polygons = "\n".join(
        f'<path class="hex" d="{svg_path(row.geometry, xy)}" data-bezirk="{escape(row.district)}" '
        f'data-intensity="{row.intensity}" data-ratio="{row.ratio}"/>'
        for row in hexes[["district", "intensity", "ratio", "geometry"]].itertuples(index=False)
    )
    color_rules = "\n".join(
        f'#map[data-layer="{layer}"] .hex[data-{layer}="{index}"] {{ fill: {color}; }}'
        for layer, colors in [("intensity", INTENSITY), ("ratio", RATIO)]
        for index, color in enumerate(colors)
    )
    subtitle = PREVIEW if preview else "2016–2025 · 3-km-Sechsecke"
    html = HTML_TEMPLATE.substitute(
        subtitle=escape(subtitle), color_rules=color_rules, state=state, polygons=polygons,
        viewbox=f"0 0 {(xmax - xmin) / 1000:.3f} {(ymax - ymin) / 1000:.3f}",
        basemap_at=embedded_basemap(background_path, bounds, xy),
        borders=borders, municipalities=municipalities, roads=roads,
        intensity_title=INTENSITY_LEGEND, ratio_title=RATIO_LEGEND,
        intensity_legend=html_legend(INTENSITY, intensity_labels),
        empty_intensity="" if intensity_labels else "<p>Keine freigegebenen Sechsecke.</p>",
        ratio_legend=html_legend(RATIO, RATIO_LABELS),
        suppression_fill=SUPPRESSION_FILL, suppression_label=escape(SUPPRESSION_LABEL),
        municipality_line=MUNICIPALITY_LINE,
        background=BACKGROUND, state_line=STATE_LINE, district_line=DISTRICT_LINE, road_line=ROAD_LINE,
    )
    path.write_text(html, encoding="utf-8")


def export_maps(hexes, basemap, intensity_labels, output, *, preview=False):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    figures = print_figures(hexes, basemap, intensity_labels, preview=preview)
    with TemporaryDirectory(prefix=".karten-", dir=output) as staging:
        folder = Path(staging)
        try:
            with plt.rc_context({"savefig.bbox": None, "pdf.fonttype": 42, "svg.fonttype": "none",
                                 "hatch.linewidth": .32}):
                for stem, figure in figures.items():
                    for suffix in ("pdf", "svg", "png"):
                        figure.savefig(folder / f"{stem}.{suffix}", dpi=600, facecolor="white")
            write_interactive(hexes, basemap, intensity_labels, folder / "standortgruendungen.html", preview=preview)
            return [path.replace(output / path.name) for path in sorted(folder.iterdir())]
        finally:
            for figure in figures.values():
                plt.close(figure)

## 5. Export

In [ ]:
cells, difference, _ = score_founding(input_paths(PROJECT_DIR))
print(f"NB2-Log-Likelihood geprüft |Δ| = {difference:.3g}")
basemap = load_basemap(PROJECT_DIR, gip_path=GIP_PBF)
hexes, intensity_labels = classify_hexes(cells, basemap, minimum_count=MINIMUM_COUNT)
del cells

for file in export_maps(hexes, basemap, intensity_labels, OUTPUT_DIR):
    print(file.relative_to(PROJECT_DIR))